In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
fact_df = pd.read_csv('apollo_appointments_fact.csv')
dim_df = pd.read_csv('apollo_doctors_dim.csv')

In [3]:
fact_df.columns

Index(['appointment_id', 'patient_id', 'doctor_id', 'appointment_date',
       'appointment_day', 'appointment_day_of_week', 'appointment_month',
       'appointment_quarter', 'appointment_year', 'appointment_hour',
       'appointment_minute', 'time_slot', 'time_of_day', 'booking_date',
       'booking_lead_days', 'booking_channel', 'appointment_type', 'specialty',
       'city', 'state', 'hospital_name', 'patient_age', 'age_group',
       'patient_gender', 'patient_city_tier', 'patient_chronic_condition',
       'apollo_member', 'membership_type', 'reminder_type', 'reminders_sent',
       'patient_prior_visits', 'patient_prior_no_shows', 'is_first_visit',
       'repeat_visit', 'visit_reason', 'appointment_status', 'no_show',
       'no_show_flag', 'cancellation_reason', 'consultation_fee',
       'discount_pct', 'actual_fee_charged', 'insurance_coverage',
       'insurance_provider', 'insurance_covered_amount',
       'patient_out_of_pocket', 'revenue_realized', 'payment_mode',
    

In [4]:
dim_df.columns

Index(['doctor_id', 'doctor_name', 'specialty', 'qualification',
       'experience_years', 'city', 'state', 'hospital_name',
       'consultation_fee', 'rating', 'total_reviews', 'available_days',
       'avg_slot_duration_min', 'accepts_insurance', 'teleconsult_enabled'],
      dtype='str')

In [5]:
fact_df['appointment_date'] = pd.to_datetime(fact_df['appointment_date'])
fact_df['booking_date'] = pd.to_datetime(fact_df['booking_date'])

In [6]:
df = pd.merge(fact_df, dim_df, on='doctor_id', how='left', suffixes=('_fact', '_dim'))

In [7]:
historical_df = df[df['appointment_status'] != 'Scheduled'].copy()
completed_df = df[df['appointment_status'] == 'Completed'].copy()

# A. Appointment Volume Trend (Monthly)
monthly_vol = historical_df.groupby(['appointment_year', 'appointment_month']).size().reset_index(name='volume')
monthly_vol['year_month'] = monthly_vol['appointment_year'].astype(str) + '-' + monthly_vol['appointment_month'].astype(str).str.zfill(2)

fig1 = px.line(monthly_vol, x='year_month', y='volume', markers=True, 
               title='Monthly Appointment Volume Trend (2022-2024)', labels={'volume': 'Total Appointments'})
fig1.show()

In [8]:
# B. Outcome Split
outcome_counts = historical_df['appointment_status'].value_counts().reset_index()
outcome_counts.columns = ['Status', 'Count']

fig2 = px.pie(outcome_counts, values='Count', names='Status', hole=0.4, 
              title='Appointment Outcome Distribution', color='Status',
              color_discrete_map={'Completed': '#2ca02c', 'No-Show': '#d62728', 'Cancelled': '#ff7f0e'})
fig2.show()

In [9]:
# C. Booking Channel Mix
channel_vol = historical_df['booking_channel'].value_counts().reset_index()
channel_vol.columns = ['Channel', 'Count']
fig3 = px.bar(channel_vol, x='Channel', y='Count', text='Count', title='Appointment Volume by Booking Channel')
fig3.update_traces(textposition='outside')
fig3.show()

In [ ]:
# NO show analysis

In [15]:
# A. No-Show Rate by Specialty & City
noshow_specialty = historical_df.groupby('specialty_fact')['no_show_flag'].mean().reset_index()
noshow_specialty['no_show_pct'] = (noshow_specialty['no_show_flag'] * 100).round(1)
noshow_specialty = noshow_specialty.sort_values('no_show_pct', ascending=False)

fig4 = px.bar(noshow_specialty.head(10), x='no_show_pct', y='specialty_fact', orientation='h',
              title='Top 10 Specialties by No-Show Rate (%)', color='no_show_pct', color_continuous_scale='Reds')
fig4.show()


In [13]:
print(historical_df.columns)

Index(['appointment_id', 'patient_id', 'doctor_id', 'appointment_date',
       'appointment_day', 'appointment_day_of_week', 'appointment_month',
       'appointment_quarter', 'appointment_year', 'appointment_hour',
       'appointment_minute', 'time_slot', 'time_of_day', 'booking_date',
       'booking_lead_days', 'booking_channel', 'appointment_type',
       'specialty_fact', 'city_fact', 'state_fact', 'hospital_name_fact',
       'patient_age', 'age_group', 'patient_gender', 'patient_city_tier',
       'patient_chronic_condition', 'apollo_member', 'membership_type',
       'reminder_type', 'reminders_sent', 'patient_prior_visits',
       'patient_prior_no_shows', 'is_first_visit', 'repeat_visit',
       'visit_reason', 'appointment_status', 'no_show', 'no_show_flag',
       'cancellation_reason', 'consultation_fee_fact', 'discount_pct',
       'actual_fee_charged', 'insurance_coverage', 'insurance_provider',
       'insurance_covered_amount', 'patient_out_of_pocket', 'revenue_realiz

In [16]:
# B. Booking Lead Time Effect
# Bin the lead days for better visualization
bins = [0, 1, 3, 7, 14, 30, 100]
labels = ['Same Day', '1-3 Days', '4-7 Days', '1-2 Weeks', '2-4 Weeks', '>1 Month']
historical_df['lead_time_bin'] = pd.cut(historical_df['booking_lead_days'], bins=bins, labels=labels, right=True)

leadtime_noshow = historical_df.groupby('lead_time_bin', observed=False)['no_show_flag'].mean().reset_index()
leadtime_noshow['no_show_pct'] = (leadtime_noshow['no_show_flag'] * 100).round(1)

fig5 = px.line(leadtime_noshow, x='lead_time_bin', y='no_show_pct', markers=True,
               title='No-Show Rate (%) by Booking Lead Time')
fig5.show()

In [17]:
# C. Time of Day & Day of Week Risk
time_noshow = historical_df.groupby('time_of_day')['no_show_flag'].mean().reset_index()
fig6 = px.bar(time_noshow, x='time_of_day', y='no_show_flag', text_auto='.1%', 
              title='No-Show Rate by Time of Day')
fig6.update_yaxes(title='No-Show Rate')
fig6.show()

In [ ]:
#Reminder and Engagement Effectiveness

In [18]:
# A. Reminder Type vs No-Show
reminder_noshow = historical_df.groupby('reminder_type')['no_show_flag'].mean().reset_index()
reminder_noshow['no_show_pct'] = (reminder_noshow['no_show_flag'] * 100).round(1)
reminder_noshow = reminder_noshow.sort_values('no_show_pct')

fig7 = px.bar(reminder_noshow, x='reminder_type', y='no_show_pct', text_auto='.1%',
              title='No-Show Rate (%) by Reminder Type', color='no_show_pct', color_continuous_scale='RdYlGn_r')
fig7.show()

In [19]:
# B. Prior No-Show History Prediction
prior_noshow_corr = historical_df['patient_prior_no_shows'].corr(historical_df['no_show_flag'])
print(f"Correlation between prior no-shows and current no-show: {prior_noshow_corr:.3f}")

Correlation between prior no-shows and current no-show: 0.060


In [20]:
# C. Member/Repeat vs Non-Member/First-Time
# Create a combined segment column
historical_df['patient_segment'] = historical_df['apollo_member'] + ' - ' + historical_df['repeat_visit']
segment_noshow = historical_df.groupby('patient_segment')['no_show_flag'].mean().reset_index()
segment_noshow['no_show_pct'] = (segment_noshow['no_show_flag'] * 100).round(1)

fig8 = px.bar(segment_noshow, x='patient_segment', y='no_show_pct', text_auto='.1%',
              title='No-Show Rate (%) by Patient Segment')
fig8.show()

In [ ]:
#Patient and Demographic Segmentation

In [22]:
# A. Age Distribution in Key Specialties
target_specs = ['Paediatrics', 'Gynaecology', 'Psychiatry']
filtered_df = historical_df[historical_df['specialty_fact'].isin(target_specs)]

fig9 = px.histogram(filtered_df, x='patient_age', color='specialty_fact', barmode='overlay', opacity=0.7,
                    title='Patient Age Distribution by Key Specialties', nbins=30)
fig9.show()

In [23]:
# B. Visit Reason No-Show Risk
reason_noshow = historical_df.groupby('visit_reason')['no_show_flag'].mean().reset_index()
reason_noshow['no_show_pct'] = (reason_noshow['no_show_flag'] * 100).round(1)
reason_noshow = reason_noshow.sort_values('no_show_pct', ascending=False)

fig10 = px.bar(reason_noshow, x='no_show_pct', y='visit_reason', orientation='h',
               title='No-Show Rate (%) by Visit Reason')
fig10.show()

In [ ]:
#Financial Performance

In [29]:
# 1. Create a flexible filter for no-shows (handles 1, '1', 'Yes', True, etc.)
is_no_show = historical_df['no_show_flag'].astype(str).str.lower().isin(['1', 'yes', 'true'])

# 2. Filter the dataframe
no_shows_df = historical_df[is_no_show]

# 3. Ensure the fee column is numeric (in case it was read as text/strings)
# pd.to_numeric with errors='coerce' turns any weird text into NaN, which .sum() safely ignores
fees_lost = pd.to_numeric(no_shows_df['actual_fee_charged'], errors='coerce')

# 4. Calculate the sum
revenue_lost = fees_lost.sum()

print(f"Total Estimated Revenue Lost to No-Shows: ₹{revenue_lost:,.2f}")
print(f"Total No-Show Appointments Analyzed: {len(no_shows_df)}")

Total Estimated Revenue Lost to No-Shows: ₹0.00
Total No-Show Appointments Analyzed: 11584


In [31]:
# B. Revenue by City and Payment Mode Mix (Completed only)
city_revenue = completed_df.groupby(['city_dim', 'payment_mode'])['revenue_realized'].sum().reset_index()

fig11 = px.bar(city_revenue, x='city_dim', y='revenue_realized', color='payment_mode', barmode='stack',
               title='Total Realized Revenue by City and Payment Mode')
fig11.show()

In [33]:
# C. Insurance Coverage Effect on Out-of-Pocket
insurance_oop = completed_df.groupby('insurance_coverage')['patient_out_of_pocket'].mean().reset_index()
fig12 = px.bar(insurance_oop, x='insurance_coverage', y='patient_out_of_pocket', text_auto='₹,.0f',
               title='Average Patient Out-of-Pocket Expense by Insurance Coverage')
fig12.show()

In [ ]:
#Doctor Utilisation and Service Quality

In [40]:
# A. Doctor Experience vs Consultation Fee (Joining Dim Data)
fig13 = px.scatter(completed_df, x='experience_years', y='consultation_fee_dim', 
                   color='specialty_dim', trendline='ols', hover_data=['doctor_name'],
                   title='Relationship Between Doctor Experience and Consultation Fee')
fig13.show()

In [35]:
print(historical_df.columns)

Index(['appointment_id', 'patient_id', 'doctor_id', 'appointment_date',
       'appointment_day', 'appointment_day_of_week', 'appointment_month',
       'appointment_quarter', 'appointment_year', 'appointment_hour',
       'appointment_minute', 'time_slot', 'time_of_day', 'booking_date',
       'booking_lead_days', 'booking_channel', 'appointment_type',
       'specialty_fact', 'city_fact', 'state_fact', 'hospital_name_fact',
       'patient_age', 'age_group', 'patient_gender', 'patient_city_tier',
       'patient_chronic_condition', 'apollo_member', 'membership_type',
       'reminder_type', 'reminders_sent', 'patient_prior_visits',
       'patient_prior_no_shows', 'is_first_visit', 'repeat_visit',
       'visit_reason', 'appointment_status', 'no_show', 'no_show_flag',
       'cancellation_reason', 'consultation_fee_fact', 'discount_pct',
       'actual_fee_charged', 'insurance_coverage', 'insurance_provider',
       'insurance_covered_amount', 'patient_out_of_pocket', 'revenue_realiz

In [42]:
# B. Wait Time vs Patient Satisfaction (Completed only, drop nulls)
clean_quality_df = completed_df.dropna(subset=['wait_time_minutes', 'patient_satisfaction_score'])

fig14 = px.scatter(clean_quality_df, x='wait_time_minutes', y='patient_satisfaction_score', 
                   color='specialty_dim', trendline='lowess', opacity=0.6,
                   title='Patient Wait Time vs Satisfaction Score (Completed Appointments)')
fig14.show()


In [44]:
# C. Doctor Utilization by Specialty
util_specialty = completed_df.groupby('specialty_dim')['doctor_utilization_pct'].mean().reset_index()
util_specialty = util_specialty.sort_values('doctor_utilization_pct', ascending=False)

fig15 = px.bar(util_specialty, x='doctor_utilization_pct', y='specialty_dim', orientation='h',
               title='Average Doctor Utilization Rate (%) by Specialty')
fig15.show()